# Taller evaluable: SQL Server y T-SQL
## Módulo 2 · Bloque de taller · Caso: biblioteca

Diplomado en Estrategias de Datos: Python, SQL Server y Big Data. Facultad de Estadística, Universidad Santo Tomás.

Este cuaderno se resuelve desde Python, conectado al motor SQL Server del Codespace. Antes de comenzar, el entorno de Python debe estar preparado y el kernel seleccionado, conforme al manual de consultas desde Python. Complete cada celda donde se indica.

> **Versión resuelta para entrega:** las celdas pendientes fueron completadas con consultas T-SQL ejecutadas desde Python mediante `pymssql` y `pandas`. El cuaderno está pensado para ejecutarse en el Codespace de la clase, donde el servicio de SQL Server está disponible como `server="db"`.

> Las salidas se generan al ejecutar el notebook completo: primero se crea la base `biblioteca`, luego se crean las tablas, se cargan los datos sintéticos y finalmente se resuelven las consultas solicitadas.


## Parte práctica

### 0. Conexión y creación de la base de datos
Las dos celdas siguientes se conectan al motor, crean la base `biblioteca` y se conectan a ella. Ejecútelas tal cual.

In [1]:
import pymssql
import pandas as pd
import warnings
warnings.filterwarnings("ignore")   # oculta avisos no criticos de pandas

# Conexion a 'master' para crear la base. Si ya existe, se fuerza el cierre
# de otras conexiones antes de borrarla, para evitar el error 'base en uso'.
con_master = pymssql.connect(server="db", user="sa",
                             password="TuClave_Fuerte123",
                             database="master", autocommit=True)
cur = con_master.cursor()
cur.execute("""
IF DB_ID('biblioteca') IS NOT NULL
BEGIN
    ALTER DATABASE biblioteca SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE biblioteca;
END
""")
cur.execute("CREATE DATABASE biblioteca")
con_master.close()
print("Base de datos biblioteca creada")

Base de datos biblioteca creada


In [2]:
# Conexion a la base biblioteca, que se usara en todo el taller
con = pymssql.connect(server="db", user="sa",
                      password="TuClave_Fuerte123",
                      database="biblioteca")
print("Conectado a biblioteca")

Conectado a biblioteca


### 1. Crear las tablas
Cree las tablas `lectores` y `prestamos` con esta especificación:

**lectores**: `lector_id` (entero, clave primaria), `nombre` (texto, obligatorio), `programa` (texto, admite nulo), `sede` (texto, obligatorio), `fecha_inscripcion` (fecha).

**prestamos**: `prestamo_id` (entero, clave primaria), `lector_id` (entero, clave foránea a `lectores`), `categoria` (texto), `titulo` (texto), `dias_prestamo` (entero), `multa` (decimal de 10 dígitos con 2 decimales), `fecha_prestamo` (fecha).

In [3]:
cur = con.cursor()

# ---------------------------------------------------------------------
# Creación de tablas del caso biblioteca
# ---------------------------------------------------------------------
# Se eliminan primero las tablas si existían previamente. Esto permite
# volver a ejecutar esta celda sin errores. El orden importa: primero se
# borra prestamos porque depende de lectores mediante una clave foránea.
cur.execute("""
IF OBJECT_ID('prestamos', 'U') IS NOT NULL DROP TABLE prestamos;
IF OBJECT_ID('lectores', 'U') IS NOT NULL DROP TABLE lectores;
""")

# Tabla de lectores: representa a las personas inscritas en la biblioteca.
# lector_id es la clave primaria porque identifica de forma única a cada lector.
cur.execute("""
CREATE TABLE lectores (
    lector_id INT NOT NULL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    programa VARCHAR(80) NULL,
    sede VARCHAR(60) NOT NULL,
    fecha_inscripcion DATE NULL
);
""")

# Tabla de préstamos: registra cada préstamo realizado.
# prestamo_id identifica cada préstamo y lector_id permite relacionarlo
# con la tabla lectores. La multa se define como DECIMAL(10,2) para manejar
# valores monetarios sin los problemas de precisión de FLOAT.
cur.execute("""
CREATE TABLE prestamos (
    prestamo_id INT NOT NULL PRIMARY KEY,
    lector_id INT NOT NULL,
    categoria VARCHAR(50) NULL,
    titulo VARCHAR(150) NULL,
    dias_prestamo INT NULL,
    multa DECIMAL(10,2) NULL,
    fecha_prestamo DATE NULL,
    CONSTRAINT FK_prestamos_lectores
        FOREIGN KEY (lector_id) REFERENCES lectores(lector_id)
);
""")

con.commit()
print("Tablas creadas: lectores y prestamos")


Tablas creadas: lectores y prestamos


### 2. Cargar los datos
Ejecute esta celda para cargar los datos de ejemplo (sintéticos e ilustrativos).

In [4]:
# Datos sinteticos e ilustrativos (no son cifras reales)
lectores = [
    (1, 'Mariana Ospina', 'Medicina', 'Bogotá', '2023-02-19'),
    (2, 'Julián Cárdenas', 'Ingeniería', 'Chía', '2023-10-13'),
    (3, 'Natalia Bermúdez', 'Medicina', 'Chía', '2023-03-16'),
    (4, 'Sebastián Arango', None, 'Chía', '2025-07-28'),
    (5, 'Carolina Peña', 'Psicología', 'Chía', '2024-03-17'),
    (6, 'Esteban Gil', 'Economía', 'Villavicencio', '2023-07-17'),
    (7, 'Laura Montoya', None, 'Bogotá', '2023-02-10'),
    (8, 'Mateo Guerrero', 'Derecho', 'Chía', '2025-04-22'),
    (9, 'Isabela Cortés', 'Derecho', 'Chía', '2025-11-01'),
    (10, 'Samuel Rincón', 'Psicología', 'Chía', '2023-10-17'),
    (11, 'Gabriela León', 'Medicina', 'Bogotá', '2023-09-18'),
    (12, 'Nicolás Acosta', 'Psicología', 'Chía', '2023-10-03'),
    (13, 'Manuela Torres', 'Medicina', 'Tunja', '2024-10-27'),
    (14, 'David Salazar', 'Derecho', 'Chía', '2025-12-26'),
    (15, 'Antonia Vega', 'Ingeniería', 'Bogotá', '2025-08-04'),
    (16, 'Emilio Cárdenas', 'Medicina', 'Chía', '2023-12-25'),
    (17, 'Renata Ochoa', 'Derecho', 'Villavicencio', '2023-09-15'),
    (18, 'Simón Duarte', 'Psicología', 'Tunja', '2023-05-14'),
]

prestamos = [
    (1, 17, 'Tecnico', 'Calculo', 39, 0, '2026-06-21'),
    (2, 7, 'Ciencia', 'Cosmos', 11, 5400, '2026-06-15'),
    (3, 10, 'Historia', 'El Bogotazo', 5, 12500, '2026-05-15'),
    (4, 14, 'Novela', 'Delirio', 6, 3200, '2026-02-17'),
    (5, 7, 'Novela', 'Los ejercitos', 15, 1500, '2026-06-28'),
    (6, 7, 'Ciencia', 'El gen egoista', 41, 12500, '2026-06-25'),
    (7, 4, 'Tecnico', 'Bases de datos', 20, 0, '2026-01-23'),
    (8, 15, 'Tecnico', 'Algebra lineal', 6, 1500, '2026-05-13'),
    (9, 2, 'Tecnico', 'Calculo', 5, 0, '2026-04-17'),
    (10, 1, 'Novela', 'Los ejercitos', 24, 12500, '2026-01-11'),
    (11, 3, 'Novela', 'El olvido que seremos', 44, 1500, '2026-06-15'),
    (12, 8, 'Ciencia', 'Sapiens', 9, 21000, '2026-01-15'),
    (13, 15, 'Historia', 'El Bogotazo', 15, 0, '2026-01-26'),
    (14, 3, 'Historia', '1810', 23, 12500, '2026-01-14'),
    (15, 4, 'Historia', 'Bolivar', 28, 21000, '2026-03-22'),
    (16, 17, 'Novela', 'Los ejercitos', 9, 0, '2026-05-10'),
    (17, 6, 'Novela', 'La voragine', 44, 0, '2026-05-04'),
    (18, 4, 'Historia', 'El Bogotazo', 8, 8000, '2026-03-20'),
    (19, 7, 'Historia', 'Historia minima', 15, 5400, '2026-01-22'),
    (20, 1, 'Historia', 'Historia minima', 27, 0, '2026-04-07'),
    (21, 1, 'Ciencia', 'El gen egoista', 36, 5400, '2026-01-13'),
    (22, 17, 'Historia', 'La violencia en Colombia', 9, 21000, '2026-02-27'),
    (23, 10, 'Ciencia', 'El origen', 5, 21000, '2026-03-22'),
    (24, 3, 'Ciencia', 'Cosmos', 40, 3200, '2026-05-07'),
    (25, 5, 'Historia', 'La violencia en Colombia', 14, 0, '2026-01-13'),
    (26, 10, 'Ciencia', 'Sapiens', 6, 0, '2026-01-06'),
    (27, 8, 'Novela', 'Cien años de soledad', 3, 5400, '2026-04-16'),
    (28, 8, 'Historia', 'La violencia en Colombia', 28, 1500, '2026-02-07'),
    (29, 11, 'Ciencia', 'El gen egoista', 5, 0, '2026-06-13'),
    (30, 7, 'Tecnico', 'Algebra lineal', 19, 8000, '2026-03-22'),
    (31, 16, 'Tecnico', 'Calculo', 10, 8000, '2026-01-04'),
    (32, 10, 'Ciencia', 'El gen egoista', 23, 5400, '2026-03-22'),
    (33, 11, 'Novela', 'Cien años de soledad', 23, 0, '2026-01-19'),
    (34, 1, 'Ciencia', 'Sapiens', 18, 0, '2026-01-09'),
    (35, 18, 'Tecnico', 'Bases de datos', 13, 0, '2026-03-28'),
    (36, 14, 'Ciencia', 'Breve historia del tiempo', 44, 3200, '2026-02-13'),
    (37, 7, 'Historia', 'El Bogotazo', 36, 3200, '2026-06-05'),
    (38, 16, 'Ciencia', 'Breve historia del tiempo', 11, 0, '2026-03-06'),
    (39, 3, 'Tecnico', 'Calculo', 17, 0, '2026-01-12'),
    (40, 11, 'Novela', 'Los ejercitos', 5, 3200, '2026-01-06'),
    (41, 16, 'Historia', 'La violencia en Colombia', 42, 5400, '2026-06-25'),
    (42, 1, 'Novela', 'Cien años de soledad', 22, 21000, '2026-02-24'),
    (43, 17, 'Tecnico', 'Calculo', 17, 1500, '2026-06-11'),
    (44, 8, 'Tecnico', 'Redes', 23, 1500, '2026-03-20'),
    (45, 1, 'Tecnico', 'Estadistica aplicada', 4, 5400, '2026-06-04'),
    (46, 12, 'Novela', 'El olvido que seremos', 24, 1500, '2026-06-28'),
    (47, 15, 'Novela', 'El olvido que seremos', 39, 1500, '2026-01-15'),
    (48, 16, 'Tecnico', 'Redes', 32, 21000, '2026-04-17'),
]

cur = con.cursor()
cur.executemany("INSERT INTO lectores VALUES (%s, %s, %s, %s, %s)", lectores)
cur.executemany("INSERT INTO prestamos VALUES (%s, %s, %s, %s, %s, %s, %s)", prestamos)
con.commit()
print("Datos cargados:", len(lectores), "lectores y", len(prestamos), "prestamos")

Datos cargados: 18 lectores y 48 prestamos


## Consultas
Resuelva cada consulta en su celda. El resultado debe mostrarse como una tabla.

### Consulta 1. SELECT y TOP
Muestre los **primeros 8 lectores**, ordenados por `sede`.

In [5]:
# Consulta 1. SELECT y TOP
# Objetivo: mostrar los primeros 8 lectores ordenados por sede.
# En SQL Server se usa TOP para limitar la cantidad de filas retornadas.

query = """
SELECT TOP (8)
    lector_id,
    nombre,
    programa,
    sede,
    fecha_inscripcion
FROM lectores
ORDER BY sede ASC, lector_id ASC;
"""

df = pd.read_sql(query, con)
df


,lector_id,nombre,programa,sede,fecha_inscripcion
0,1,Mariana Ospina,Medicina,Bogotá,2023-02-19
1,7,Laura Montoya,None,Bogotá,2023-02-10
2,11,Gabriela León,Medicina,Bogotá,2023-09-18
3,15,Antonia Vega,Ingeniería,Bogotá,2025-08-04
4,2,Julián Cárdenas,Ingeniería,Chía,2023-10-13
5,3,Natalia Bermúdez,Medicina,Chía,2023-03-16
6,4,Sebastián Arango,None,Chía,2025-07-28
7,5,Carolina Peña,Psicología,Chía,2024-03-17


### Consulta 2. Variables y filtro
Defina en Python una variable `dias_minimos = 30` e insértela con una f-string. Muestre los préstamos cuyo `dias_prestamo` es **mayor o igual** al mínimo, ordenados de mayor a menor por días.

In [6]:
# Consulta 2. Variables y filtro
# Objetivo: filtrar préstamos con duración mayor o igual a un mínimo definido en Python.
# El enunciado pide insertar la variable usando una f-string.

dias_minimos = 30

query = f"""
SELECT
    prestamo_id,
    lector_id,
    categoria,
    titulo,
    dias_prestamo,
    multa,
    fecha_prestamo
FROM prestamos
WHERE dias_prestamo >= {dias_minimos}
ORDER BY dias_prestamo DESC, multa DESC, prestamo_id ASC;
"""

df = pd.read_sql(query, con)
df


,prestamo_id,lector_id,categoria,titulo,dias_prestamo,multa,fecha_prestamo
0,36,14,Ciencia,Breve historia del tiempo,44,3200,2026-02-13
1,11,3,Novela,El olvido que seremos,44,1500,2026-06-15
2,17,6,Novela,La voragine,44,0,2026-05-04
3,41,16,Historia,La violencia en Colombia,42,5400,2026-06-25
4,6,7,Ciencia,El gen egoista,41,12500,2026-06-25
5,24,3,Ciencia,Cosmos,40,3200,2026-05-07
6,47,15,Novela,El olvido que seremos,39,1500,2026-01-15
7,1,17,Tecnico,Calculo,39,0,2026-06-21
8,21,1,Ciencia,El gen egoista,36,5400,2026-01-13
9,37,7,Historia,El Bogotazo,36,3200,2026-06-05


### Consulta 3. Limitar y paginar (TOP y OFFSET-FETCH)
Muestre los **4 préstamos con mayor multa** y, en otra tabla, los **4 siguientes** (paginación).

In [7]:
# Consulta 3. Limitar y paginar (TOP y OFFSET-FETCH)
# Objetivo: obtener primero los 4 préstamos con mayor multa y luego los 4 siguientes.
# Para que la paginación sea estable, el ORDER BY incluye prestamo_id como criterio secundario.

query_top_4 = """
SELECT TOP (4)
    prestamo_id,
    lector_id,
    categoria,
    titulo,
    dias_prestamo,
    multa,
    fecha_prestamo
FROM prestamos
ORDER BY multa DESC, prestamo_id ASC;
"""

df_top_4 = pd.read_sql(query_top_4, con)
print("Primeros 4 préstamos con mayor multa")
display(df_top_4)

query_siguientes_4 = """
SELECT
    prestamo_id,
    lector_id,
    categoria,
    titulo,
    dias_prestamo,
    multa,
    fecha_prestamo
FROM prestamos
ORDER BY multa DESC, prestamo_id ASC
OFFSET 4 ROWS FETCH NEXT 4 ROWS ONLY;
"""

df_siguientes_4 = pd.read_sql(query_siguientes_4, con)
print("Siguientes 4 préstamos según la misma ordenación")
display(df_siguientes_4)


Primeros 4 préstamos con mayor multa


,prestamo_id,lector_id,categoria,titulo,dias_prestamo,multa,fecha_prestamo
0,12,8,Ciencia,Sapiens,9,21000,2026-01-15
1,15,4,Historia,Bolivar,28,21000,2026-03-22
2,22,17,Historia,La violencia en Colombia,9,21000,2026-02-27
3,23,10,Ciencia,El origen,5,21000,2026-03-22


Siguientes 4 préstamos según la misma ordenación


,prestamo_id,lector_id,categoria,titulo,dias_prestamo,multa,fecha_prestamo
0,42,1,Novela,Cien años de soledad,22,21000,2026-02-24
1,48,16,Tecnico,Redes,32,21000,2026-04-17
2,3,10,Historia,El Bogotazo,5,12500,2026-05-15
3,6,7,Ciencia,El gen egoista,41,12500,2026-06-25


### Consulta 4. Funciones de fecha
Muestre la fecha actual, la fecha dentro de **15 días** y los días transcurridos desde el **1 de marzo de 2026**. En otra tabla, el número de préstamos y el **promedio de días** por mes.

In [8]:
# Consulta 4. Funciones de fecha
# Objetivo: usar funciones propias de T-SQL para trabajar con fechas.
# GETDATE obtiene la fecha y hora actual del servidor; CAST(... AS DATE) deja solo la fecha.

query_fechas = """
SELECT
    CAST(GETDATE() AS DATE) AS fecha_actual,
    DATEADD(DAY, 15, CAST(GETDATE() AS DATE)) AS fecha_mas_15_dias,
    DATEDIFF(DAY, CAST('2026-03-01' AS DATE), CAST(GETDATE() AS DATE)) AS dias_desde_2026_03_01;
"""

df_fechas = pd.read_sql(query_fechas, con)
print("Funciones de fecha")
display(df_fechas)

# Resumen mensual de préstamos. Se agrupa por año y mes para evitar mezclar
# meses iguales de años diferentes, una buena práctica en análisis temporal.
query_resumen_mes = """
SELECT
    YEAR(fecha_prestamo) AS anio,
    MONTH(fecha_prestamo) AS mes,
    COUNT(*) AS numero_prestamos,
    CAST(AVG(CAST(dias_prestamo AS DECIMAL(10,2))) AS DECIMAL(10,2)) AS promedio_dias
FROM prestamos
GROUP BY YEAR(fecha_prestamo), MONTH(fecha_prestamo)
ORDER BY anio ASC, mes ASC;
"""

df_resumen_mes = pd.read_sql(query_resumen_mes, con)
print("Número de préstamos y promedio de días por mes")
display(df_resumen_mes)


Funciones de fecha


,fecha_actual,fecha_mas_15_dias,dias_desde_2026_03_01
0,2026-07-01,2026-07-16,122


Número de préstamos y promedio de días por mes


,anio,mes,numero_prestamos,promedio_dias
0,2026,1,15,18.27
1,2026,2,5,21.80
2,2026,3,8,16.25
3,2026,4,4,16.75
4,2026,5,5,20.80
5,2026,6,11,25.27


### Consulta 5. Funciones de texto
Para cada lector, muestre el nombre, el nombre en **minúsculas**, su longitud y el **apellido** (la parte que va después del primer espacio).

In [9]:
# Consulta 5. Funciones de texto
# Objetivo: transformar cadenas de texto y extraer el apellido.
# LOWER convierte a minúsculas, LEN cuenta caracteres y CHARINDEX ubica el primer espacio.

query = """
SELECT
    lector_id,
    nombre,
    LOWER(nombre) AS nombre_minusculas,
    LEN(nombre) AS longitud_nombre,
    CASE
        WHEN CHARINDEX(' ', nombre) > 0
            THEN SUBSTRING(nombre, CHARINDEX(' ', nombre) + 1, LEN(nombre))
        ELSE NULL
    END AS apellido
FROM lectores
ORDER BY lector_id ASC;
"""

df = pd.read_sql(query, con)
df


,lector_id,nombre,nombre_minusculas,longitud_nombre,apellido
0,1,Mariana Ospina,mariana ospina,14,Ospina
1,2,Julián Cárdenas,julián cárdenas,15,Cárdenas
2,3,Natalia Bermúdez,natalia bermúdez,16,Bermúdez
3,4,Sebastián Arango,sebastián arango,16,Arango
4,5,Carolina Peña,carolina peña,13,Peña
5,6,Esteban Gil,esteban gil,11,Gil
6,7,Laura Montoya,laura montoya,13,Montoya
7,8,Mateo Guerrero,mateo guerrero,14,Guerrero
8,9,Isabela Cortés,isabela cortés,14,Cortés
9,10,Samuel Rincón,samuel rincón,13,Rincón


### Consulta 6. Conversión de tipos
Muestre la `multa` convertida a entero y la `fecha_prestamo` como texto en formato **dd.mm.aaaa** (estilo 104), para los 5 primeros préstamos. En otra tabla, demuestre `TRY_CONVERT` con '2024' y con 'x12'.

In [10]:
# Consulta 6. Conversión de tipos
# Objetivo: aplicar conversiones explícitas con CONVERT y conversiones seguras con TRY_CONVERT.
# El estilo 104 muestra fechas en formato dd.mm.aaaa en SQL Server.

query_conversiones = """
SELECT TOP (5)
    prestamo_id,
    multa,
    CONVERT(INT, multa) AS multa_entero,
    fecha_prestamo,
    CONVERT(VARCHAR(10), fecha_prestamo, 104) AS fecha_texto_dd_mm_aaaa
FROM prestamos
ORDER BY prestamo_id ASC;
"""

df_conversiones = pd.read_sql(query_conversiones, con)
print("Conversión de multa y fecha")
display(df_conversiones)

# TRY_CONVERT devuelve NULL cuando la conversión no es posible, en lugar de detener la consulta.
query_try_convert = """
SELECT
    valor_original,
    TRY_CONVERT(INT, valor_original) AS valor_convertido_a_entero
FROM (VALUES ('2024'), ('x12')) AS t(valor_original);
"""

df_try_convert = pd.read_sql(query_try_convert, con)
print("Demostración de TRY_CONVERT")
display(df_try_convert)


Conversión de multa y fecha


,prestamo_id,multa,multa_entero,fecha_prestamo,fecha_texto_dd_mm_aaaa
0,1,0,0,2026-06-21,21.06.2026
1,2,5400,5400,2026-06-15,15.06.2026
2,3,12500,12500,2026-05-15,15.05.2026
3,4,3200,3200,2026-02-17,17.02.2026
4,5,1500,1500,2026-06-28,28.06.2026


Demostración de TRY_CONVERT


,valor_original,valor_convertido_a_entero
0,2024,2024.0
1,x12,NaN


### Consulta 7. Manejo de nulos
Muestre el nombre, el `programa` original y el programa corregido, reemplazando los nulos por **'No registrado'** con `COALESCE`.

In [11]:
# Consulta 7. Manejo de nulos
# Objetivo: reemplazar valores nulos de programa por una etiqueta legible.
# COALESCE devuelve el primer valor no nulo de la lista.

query = """
SELECT
    lector_id,
    nombre,
    programa AS programa_original,
    COALESCE(programa, 'No registrado') AS programa_corregido
FROM lectores
ORDER BY lector_id ASC;
"""

df = pd.read_sql(query, con)
df


,lector_id,nombre,programa_original,programa_corregido
0,1,Mariana Ospina,Medicina,Medicina
1,2,Julián Cárdenas,Ingeniería,Ingeniería
2,3,Natalia Bermúdez,Medicina,Medicina
3,4,Sebastián Arango,None,No registrado
4,5,Carolina Peña,Psicología,Psicología
5,6,Esteban Gil,Economía,Economía
6,7,Laura Montoya,None,No registrado
7,8,Mateo Guerrero,Derecho,Derecho
8,9,Isabela Cortés,Derecho,Derecho
9,10,Samuel Rincón,Psicología,Psicología


### Consulta 8. Lógica condicional (CASE e IIF)
Clasifique cada préstamo con `IIF` en 'Con multa' o 'Sin multa' según la multa sea mayor que cero, y con `CASE` la duración en 'Largo' (30 días o más), 'Medio' (15 a 29) o 'Corto' (menos de 15).

In [12]:
# Consulta 8. Lógica condicional (CASE e IIF)
# Objetivo: crear variables categóricas a partir de condiciones.
# IIF se usa para una condición simple; CASE permite múltiples tramos de clasificación.

query = """
SELECT
    prestamo_id,
    lector_id,
    categoria,
    titulo,
    dias_prestamo,
    multa,
    IIF(multa > 0, 'Con multa', 'Sin multa') AS estado_multa,
    CASE
        WHEN dias_prestamo >= 30 THEN 'Largo'
        WHEN dias_prestamo BETWEEN 15 AND 29 THEN 'Medio'
        ELSE 'Corto'
    END AS clasificacion_duracion
FROM prestamos
ORDER BY prestamo_id ASC;
"""

df = pd.read_sql(query, con)
df


,prestamo_id,lector_id,categoria,titulo,dias_prestamo,multa,estado_multa,clasificacion_duracion
0,1,17,Tecnico,Calculo,39,0,Sin multa,Largo
1,2,7,Ciencia,Cosmos,11,5400,Con multa,Corto
2,3,10,Historia,El Bogotazo,5,12500,Con multa,Corto
3,4,14,Novela,Delirio,6,3200,Con multa,Corto
4,5,7,Novela,Los ejercitos,15,1500,Con multa,Medio
5,6,7,Ciencia,El gen egoista,41,12500,Con multa,Largo
6,7,4,Tecnico,Bases de datos,20,0,Sin multa,Medio
7,8,15,Tecnico,Algebra lineal,6,1500,Con multa,Corto
8,9,2,Tecnico,Calculo,5,0,Sin multa,Corto
9,10,1,Novela,Los ejercitos,24,12500,Con multa,Medio


### Consulta 9. Expresiones de tabla comunes (CTE)
Con una CTE, calcule por categoría el **total de multa**, el número de préstamos y el **promedio de días**. Muestre solo las categorías cuyo total de multa supera **30000**.

In [13]:
# Consulta 9. Expresiones de tabla comunes (CTE)
# Objetivo: resumir préstamos por categoría y filtrar con base en el total de multa.
# La CTE mejora la legibilidad porque separa el cálculo del filtro final.

query = """
WITH resumen_categoria AS (
    SELECT
        categoria,
        SUM(multa) AS total_multa,
        COUNT(*) AS numero_prestamos,
        AVG(CAST(dias_prestamo AS DECIMAL(10,2))) AS promedio_dias
    FROM prestamos
    GROUP BY categoria
)
SELECT
    categoria,
    CAST(total_multa AS DECIMAL(12,2)) AS total_multa,
    numero_prestamos,
    CAST(promedio_dias AS DECIMAL(10,2)) AS promedio_dias
FROM resumen_categoria
WHERE total_multa > 30000
ORDER BY total_multa DESC;
"""

df = pd.read_sql(query, con)
df


,categoria,total_multa,numero_prestamos,promedio_dias
0,Historia,90500,12,20.83
1,Ciencia,77100,12,20.75
2,Novela,51300,12,21.50
3,Tecnico,46900,12,17.08


### Consulta 10. Funciones de ventana (OVER)
Con `ROW_NUMBER` y `PARTITION BY`, obtenga los **2 préstamos con mayor multa de cada categoría**.

In [14]:
# Consulta 10. Funciones de ventana (OVER)
# Objetivo: obtener los 2 préstamos con mayor multa dentro de cada categoría.
# ROW_NUMBER numera las filas dentro de cada grupo definido por PARTITION BY.

query = """
WITH prestamos_ordenados AS (
    SELECT
        p.prestamo_id,
        p.lector_id,
        l.nombre,
        p.categoria,
        p.titulo,
        p.dias_prestamo,
        p.multa,
        p.fecha_prestamo,
        ROW_NUMBER() OVER (
            PARTITION BY p.categoria
            ORDER BY p.multa DESC, p.prestamo_id ASC
        ) AS posicion_en_categoria
    FROM prestamos AS p
    INNER JOIN lectores AS l
        ON p.lector_id = l.lector_id
)
SELECT
    categoria,
    posicion_en_categoria,
    prestamo_id,
    lector_id,
    nombre,
    titulo,
    dias_prestamo,
    multa,
    fecha_prestamo
FROM prestamos_ordenados
WHERE posicion_en_categoria <= 2
ORDER BY categoria ASC, posicion_en_categoria ASC;
"""

df = pd.read_sql(query, con)
df


,categoria,posicion_en_categoria,prestamo_id,lector_id,nombre,titulo,dias_prestamo,multa,fecha_prestamo
0,Ciencia,1,12,8,Mateo Guerrero,Sapiens,9,21000,2026-01-15
1,Ciencia,2,23,10,Samuel Rincón,El origen,5,21000,2026-03-22
2,Historia,1,15,4,Sebastián Arango,Bolivar,28,21000,2026-03-22
3,Historia,2,22,17,Renata Ochoa,La violencia en Colombia,9,21000,2026-02-27
4,Novela,1,42,1,Mariana Ospina,Cien años de soledad,22,21000,2026-02-24
5,Novela,2,10,1,Mariana Ospina,Los ejercitos,24,12500,2026-01-11
6,Tecnico,1,48,16,Emilio Cárdenas,Redes,32,21000,2026-04-17
7,Tecnico,2,30,7,Laura Montoya,Algebra lineal,19,8000,2026-03-22


### Cierre
Al terminar, cierre la conexión con el motor.

In [15]:
con.close()
print("Conexión cerrada")

Conexión cerrada
